# VAE Math

Part of **ML Math Applied**.

## 1. Intuition First

A variational autoencoder compresses data into a probabilistic latent code and pays a KL penalty when that code drifts too far from a simple prior.

![VAE ELBO](../assets/diagrams/vae_elbo.svg)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5)

## 2. ELBO and Diagonal-Gaussian KL

For encoder posterior $q_\phi(z \mid x)$ and decoder likelihood $p_\theta(x \mid z)$:

$$
\log p_\theta(x)
\geq
\mathbb{E}_{q_\phi(z \mid x)} [\log p_\theta(x \mid z)]
-
\mathrm{KL}(q_\phi(z \mid x) \| p(z)).
$$

If $q_\phi(z \mid x) = \mathcal{N}(\mu, \operatorname{diag}(\sigma^2))$ and $p(z) = \mathcal{N}(0, I)$, then

$$
\mathrm{KL} = \frac{1}{2} \sum_j \left(\exp(\log \sigma_j^2) + \mu_j^2 - 1 - \log \sigma_j^2\right).
$$

Sampling uses the reparameterization trick:

$$
z = \mu + \sigma \odot \epsilon,
\qquad
\epsilon \sim \mathcal{N}(0, I).
$$

In [ ]:
from matplotlib.patches import Ellipse
from src.ml_components import reparameterize_gaussian, vae_kl_divergence

mean = np.array([[1.2, -0.4], [0.2, 0.7], [-0.8, 0.3]])
log_variance = np.log(np.array([[0.25, 1.4], [1.0, 0.5], [0.7, 0.2]]))
epsilon = np.array([[0.0, 1.0], [-1.0, 0.5], [0.3, -0.2]])

latent_samples = reparameterize_gaussian(mean, log_variance, noise=epsilon)
kl_terms = vae_kl_divergence(mean, log_variance)

beta_values = np.array([0.5, 1.0, 2.0, 4.0])
reconstruction_error = np.array([0.32, 0.35, 0.42, 0.57])
beta_elbo = reconstruction_error + beta_values * kl_terms.mean()

print("latent samples =\n", latent_samples)
print("KL terms =", kl_terms)

## 3. PyTorch Verification

Compare the closed-form KL against the same expression built in PyTorch.

In [ ]:
mean_t = torch.tensor(mean, dtype=torch.float64)
log_variance_t = torch.tensor(log_variance, dtype=torch.float64)
std_t = torch.exp(0.5 * log_variance_t)
kl_t = 0.5 * torch.sum(torch.exp(log_variance_t) + mean_t**2 - 1.0 - log_variance_t, dim=-1)
latent_t = mean_t + std_t * torch.tensor(epsilon, dtype=torch.float64)

print(torch.allclose(kl_t, torch.tensor(kl_terms), atol=1e-8))
print(torch.allclose(latent_t, torch.tensor(latent_samples), atol=1e-8))

## 4. Custom Figure

The ellipse visualizes one posterior covariance and the line plot shows how increasing beta trades reconstruction quality for stronger regularization.

In [ ]:
covariance = np.diag(np.exp(log_variance[0]))
widths = 2.0 * np.sqrt(np.diag(covariance))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(latent_samples[:, 0], latent_samples[:, 1], color="tab:blue", label="samples")
ellipse = Ellipse(xy=mean[0], width=widths[0], height=widths[1], angle=0.0, facecolor="none", edgecolor="tab:red", linewidth=2)
axes[0].add_patch(ellipse)
axes[0].scatter(mean[:, 0], mean[:, 1], color="tab:orange", marker="x", s=80, label="means")
axes[0].set_title("Latent posterior geometry")
axes[0].set_xlabel("z1")
axes[0].set_ylabel("z2")
axes[0].legend()

axes[1].plot(beta_values, beta_elbo, marker="o", linewidth=2.5, color="tab:purple")
axes[1].set_title("beta-VAE objective proxy")
axes[1].set_xlabel("beta")
axes[1].set_ylabel("reconstruction + beta * KL")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Case Study: beta-VAE Tradeoff

Larger beta values push the posterior closer to the unit Gaussian prior.
That usually improves latent structure, but it also makes reconstruction harder because the code is more tightly constrained.